#### Import Libraries

In [1]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

C:\Users\DELL\AppData\Local\Temp\ipykernel_15300\2259354619.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


#### Load Environment

In [2]:
load_dotenv()
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.3)

#### Load PDF

In [3]:
loader=PyPDFLoader(r"F:\GenAI\datasets\1.Introduction to sequential data and RNNs.pdf")
docs=loader.load()

#### Chunking

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks=splitter.split_documents(docs)

#### Embeddings + FAISS

In [5]:
emb=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs=FAISS.from_documents(chunks,emb)

C:\Users\DELL\AppData\Local\Temp\ipykernel_15300\1469460087.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  emb=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### Retriever

In [6]:
retriever=vs.as_retriever(search_type="mmr",search_kwargs={"k":3})

#### Prompt

In [7]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)
prompt=ChatPromptTemplate.from_template("""Use context and mention source/page if available.
Context:{context}
Question:{question}""")

#### Generate Response

In [8]:
rag_chain=({"context":retriever|format_docs,"question":RunnablePassthrough()}|prompt|llm|StrOutputParser())
resp=rag_chain.invoke("What is RNN?")
print(resp)

results=vs.similarity_search_with_score("What is RNN?",k=3)
for d,s in results:
    print(d.metadata,s)

A Recurrent Neural Network (RNN) is a special type of neural network designed to process sequential data, where the order of information is important. Unlike traditional feedforward networks, RNNs have connections that loop back, allowing information from previous steps to influence the current output. This looping structure helps the network retain a form of memory, enabling it to learn from the sequence.

(Source: Provided context information)
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2025-10-06T14:31:20+05:30', 'author': 'Angshuraj Gharami', 'moddate': '2025-11-07T11:20:36+05:30', 'source': 'F:\\GenAI\\datasets\\1.Introduction to sequential data and RNNs.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'} 0.553423
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2025-10-06T14:31:20+05:30', 'author': 'Angshuraj Gharami', 'moddate': '2025-11-07T11:20:36+05:30', 'source': 'F:\\GenAI\\datasets\\1.In